# 01 · 编程基础：五个动词

对应讲义第 04 部分 · 约 40 分钟 · 前置：00 环境检查

所有量子程序都由五个动作组成：**准备**（初始化量子态）、**演化**（作用量子门）、**组合**（搭建线路）、**测量**（采样得到比特串）、**估计**（计算期望值与梯度）。本节逐一练习，最后用它们写出第一个完整的量子算法。

**学完本节，你将能够**
- 用模拟器制备、演化和读取量子态；
- 用 `Circuit` 搭建线路，并按小端序正确读出比特串；
- 通过采样与期望值得到结果，并计算参数梯度；
- 独立写出 Bernstein–Vazirani 算法。

**怎么用**：从上到下依次运行。「练习」单元格中的 `...` 需要替换成你的代码；运行后，下面的 `check` 会立即告诉你对错，并给出提示（不会给出答案）。课堂上建议完成练习 1、2、6，其余课后完成。参考答案在 `solutions/` 目录。

In [ ]:
import numpy as np
from mindquantum import Circuit, H, X, Z, RX, RY, RZ, UN, Measure
from mindquantum import Simulator, Hamiltonian, QubitOperator
from mqlab import check

## 1. 准备：模拟器保存量子态

`Simulator('mqvector', n)` 创建一个 n 比特的态矢量模拟器，初态为 $|0\cdots0\rangle$。`set_qs` 可以把量子态设为任意向量（自动归一化），`get_qs` 读取振幅。

In [ ]:
sim = Simulator('mqvector', 1)   # 后端 + 比特数
print(sim)

In [ ]:
sim.set_qs(np.array([1, 1]))     # 自动归一化为 (|0⟩ + |1⟩)/√2
print(sim.get_qs())              # 振幅向量
print(sim.get_qs(ket=True))      # 狄拉克记号

模拟器能直接“看见”量子态，真机做不到——这是调试量子程序最大的优势。

## 2. 演化：量子门就是幺正矩阵

每个量子门都有 `matrix()` 方法。`.on(目标, 控制)` 指定门作用在哪个比特上：**目标在前，控制在后**（与 Qiskit 的 `cx(控制, 目标)` 正好相反）。

In [ ]:
print(X.matrix())
print(H.matrix().round(3))
print(RX(np.pi / 2).matrix().round(3))

In [ ]:
sim = Simulator('mqvector', 2)
sim.apply_gate(H.on(0))          # H 作用在 q0
sim.apply_gate(X.on(1, 0))       # 目标 q1，控制 q0（CNOT）
print(sim.get_qs(ket=True))

### 练习 1 · 制备 |−⟩

在 q0 上制备 $|-\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$，可以使用 X、H、Z 的任意组合。

In [ ]:
circ = Circuit()
# 练习 1：用 += 添加量子门
...
check.minus_state(circ)

## 3. 组合：用 Circuit 把门组成线路

线路与模拟器是分开的：同一个 `Circuit` 可以交给不同后端运行，也可以打印、画图（`circ.svg()`）、求矩阵。

In [ ]:
circ = Circuit()
circ += H.on(0)
circ += X.on(1, 0)
circ += RY(np.pi / 4).on(2, 1)   # 受控 RY：目标 q2，控制 q1
circ.svg()

In [ ]:
sim = Simulator('mqvector', circ.n_qubits)
sim.apply_circuit(circ)
print(sim.get_qs(ket=True))

### 比特序：先预测，再运行

3 个比特，只对 q0 作用 X。模拟器会输出 `|100⟩` 还是 `|001⟩`？先写下你的答案，再运行下面的单元格。

In [ ]:
sim = Simulator('mqvector', 3)
sim.apply_gate(X.on(0))          # 只翻转 q0
print(sim.get_qs(ket=True))

MindSpore Quantum 使用**小端序**：比特串从右往左读，最右边是 q0，与二进制数“低位在右”一致。Qiskit 同样是小端序；很多教材和 PennyLane 把 q0 写在最左。比特序是跨框架、跨教材最常见的错误来源。

### 练习 2 · 四个 Bell 态

补全 `bell(kind)`，返回制备对应 Bell 态的两比特线路：

| kind | 量子态 |
|---|---|
| `'phi+'` | $(\lvert 00\rangle + \lvert 11\rangle)/\sqrt{2}$ |
| `'phi-'` | $(\lvert 00\rangle - \lvert 11\rangle)/\sqrt{2}$ |
| `'psi+'` | $(\lvert 01\rangle + \lvert 10\rangle)/\sqrt{2}$ |
| `'psi-'` | $(\lvert 01\rangle - \lvert 10\rangle)/\sqrt{2}$ |

提示：从 $\Phi^+$ 的线路（H + CNOT）出发，想一想在 H 之前加什么门，能改变两个比特的关系或相对相位。

In [ ]:
def bell(kind):
    circ = Circuit()
    # 练习 2：根据 kind 搭建线路
    ...
    return circ

check.bell_states(bell)

### 练习 3 · n 比特 GHZ 态

补全 `ghz(n)`，制备 $(|0\cdots0\rangle + |1\cdots1\rangle)/\sqrt{2}$。

In [ ]:
def ghz(n):
    circ = Circuit()
    # 练习 3
    ...
    return circ

check.ghz(ghz)

## 4. 测量：多次采样，得到比特串的分布

`measure_all()` 给每个比特加上测量门，`sampling` 运行 `shots` 次并统计结果；`seed` 固定随机数，便于复现。

先预测：对 3 比特 GHZ 态采样 1,000 次，会出现哪些比特串？

In [ ]:
circ = Circuit([H.on(0), X.on(1, 0), X.on(2, 1)])   # 3 比特 GHZ 态
circ.measure_all()
sim = Simulator('mqvector', 3)
res = sim.sampling(circ, shots=1000, seed=2026)
print(res.data)

两点注意：

- `sampling` 从模拟器的**当前态**出发运行线路。如果之前用 `apply_circuit` 改变过模拟器，先 `sim.reset()`，否则分布是错的；
- 采样的统计误差按 $1/\sqrt{N}$ 下降：精度提高 10 倍，需要 100 倍的采样次数。

In [ ]:
bell_m = Circuit([H.on(0), X.on(1, 0)])
bell_m.measure_all()
sim = Simulator('mqvector', 2)
for shots in [100, 10_000]:
    p00 = sim.sampling(bell_m, shots=shots, seed=1).data.get('00', 0) / shots
    print(f'shots = {shots:>6}   P(00) ≈ {p00:.3f}   理论 0.5，误差量级 {0.5 / np.sqrt(shots):.3f}')

## 5. 估计：期望值、参数化线路与梯度

期望值 $\langle\psi|H|\psi\rangle$ 是测量结果的“平均”，也是变分算法的燃料。模拟器可以精确计算；真机上要靠大量采样来估计。

In [ ]:
phi_plus = Circuit([H.on(0), X.on(1, 0)])
sim = Simulator('mqvector', 2)
for term in ['Z0', 'Z1', 'Z0 Z1']:
    ham = Hamiltonian(QubitOperator(term))
    e = sim.get_expectation(ham, phi_plus)
    print(f'<{term}> = {e.real:+.3f}')

单看每个比特，0、1 各一半，期望为 0；两个比特的乘积却总是 +1——完全相关。

### 练习 4 · X、Y 方向的关联

计算 $\Phi^+$ 上的 $\langle X_0 X_1\rangle$ 和 $\langle Y_0 Y_1\rangle$。先猜一猜，再计算。

In [ ]:
# 练习 4：仿照上面的写法计算（get_expectation 返回复数，取 .real）
xx = ...
yy = ...
check.bell_correlators(xx, yy)

### 参数化线路：参数先用名字占位，运行时再赋值

In [ ]:
circ = Circuit([RX('theta').on(0)])   # 参数用名字占位
print(circ.params_name)

ham = Hamiltonian(QubitOperator('Z0'))
sim = Simulator('mqvector', 1)
for t in [0, np.pi / 2, np.pi]:
    e = sim.get_expectation(ham, circ, pr={'theta': t})
    print(f'theta = {t:.3f}   <Z> = {e.real:+.3f}')

### 梯度：一次给出期望值与导数

`get_expectation_with_grad` 返回一个函数：输入参数值，输出期望值和对每个参数的导数。这是变分算法（实验 C、D）的训练引擎。

In [ ]:
grad_ops = sim.get_expectation_with_grad(ham, circ)
f, g = grad_ops(np.array([np.pi / 3]))     # θ = π/3
print(f'f = {f.real.item():.3f}')          # cos(π/3)
print(f'df/dθ = {g.real.item():.3f}')      # −sin(π/3)

### 练习 5 · 梯度下降

补全 `update`：返回沿梯度反方向走一步后的 θ（学习率为 `eta`）。检查程序会用它从 θ = 0.5 出发迭代 60 步，看能否找到 $\langle Z\rangle$ 的最小值 −1。

In [ ]:
def update(theta, grad, eta):
    # 练习 5
    return ...

check.gradient_descent(update)

## 6. 第一个完整算法：Bernstein–Vazirani

黑盒函数 $f(x) = s\cdot x \bmod 2$ 藏着一个 n 位秘密串 s。经典算法每次查询只能得到 s 的 1 位，需要 n 次；量子算法只需 **1 次**。

线路分四步（数据比特 $q_0 \ldots q_{n-1}$，辅助比特 $q_n$）：

1. 辅助比特先用 X 翻到 $|1\rangle$，再和所有数据比特一起作用 H，辅助比特变成 $|-\rangle$；
2. Oracle：s 的第 i 位为 1，就从 $q_i$ 连一个 CNOT 到辅助比特；
3. 数据比特再作用一次 H；
4. 测量数据比特。

原理是**相位反冲**：辅助比特处于 $|-\rangle$ 时，CNOT 的作用变成给 $|x\rangle$ 乘上 $(-1)^{s\cdot x}$，最后一层 H 把这些相位干涉成 $|s\rangle$。

注意比特序：s 按 $q_{n-1}\cdots q_0$ 书写，s 的最右一位对应 $q_0$。

### 练习 6 · 实现 BV 线路

In [ ]:
def bv_circuit(s):
    n = len(s)
    circ = Circuit()
    # 练习 6：按上面四步搭建线路（UN(H, k) 在 q0…q(k−1) 上各作用一个 H）
    ...
    return circ

check.bv(bv_circuit)

通过检查后运行下面的单元格：只采样 **1 次**，就读出了秘密串。

In [ ]:
s = '1011'
circ = bv_circuit(s)
if len(circ) == 0:
    print('请先完成练习 6')
else:
    circ = circ.remove_measure() + UN(Measure(), len(s))   # 只测量数据比特
    sim = Simulator('mqvector', circ.n_qubits)
    print(sim.sampling(circ, shots=1).data)

再试一次：把 s 改成 `'0110'` 或一个 8 位的秘密串。改之前先预测：Oracle 会连哪几个 CNOT？

## 常见错误清单

| 现象 | 原因 | 正确做法 |
|---|---|---|
| CNOT 方向反了 | 按 Qiskit 习惯写成 `X.on(控制, 目标)` | `X.on(目标, 控制)` |
| 比特串读反了 | 以为 q0 在最左 | 小端序：q0 在最右 |
| 第二次采样结果不对 | 模拟器保留了上一次的末态 | `sim.reset()` 后再运行 |
| 报错缺少参数 | `pr` 字典漏了某个参数名 | 用 `circ.params_name` 核对 |
| 一次采样就下结论 | 结果本身是随机的 | 先用 `get_qs()` 看理想态，再采样 |

## 课后挑战 · IQP 编码线路

按下图搭建 4 比特线路，参数名依次为 `alpha0` … `alpha6`：

![IQP 编码线路](images/iqp.svg)

然后：

1. 随机给 7 个参数赋值，打印线路运行后的量子态；
2. 采样 10,000 次，观察分布；
3. 思考：为什么量子态看起来毫无规律，采样分布却几乎均匀？如果在线路末尾给每个比特再加一个 H，分布还均匀吗？

这条线路正是实验 D 中把数据写进量子态的“编码线路”。

In [ ]:
circ = Circuit()
# 挑战：按图搭建线路
...
check.iqp(circ)

In [ ]:
if len(circ) == 0:
    print('请先完成挑战')
else:
    circ = circ.remove_measure()
    rng = np.random.default_rng(0)
    pr = dict(zip(circ.params_name, rng.uniform(0, 2 * np.pi, len(circ.params_name))))
    sim = Simulator('mqvector', 4)
    sim.apply_circuit(circ, pr=pr)
    print(sim.get_qs(ket=True))
    sim.reset()
    print(sim.sampling(circ + UN(Measure(), 4), pr=pr, shots=10000, seed=1).data)